In [4]:
import pandas as pd
import numpy as np
import matplotlib as plt
import seaborn as sns
import os

In [10]:
orders = pd.read_csv("../data/orders.csv")
products = pd.read_csv("../data/products.csv")
returns = pd.read_csv("../data/returns.csv")
web_traffic = pd.read_csv("../data/web_traffic.csv")
web_traffic.head(10)

,date,sessions,unique_visitors,page_views,bounce_rate,avg_session_duration_sec,traffic_source
0,2013-01-01,9760,7253,39093,0.00514,102.9,organic_search
1,2013-01-02,10456,8151,47611,0.00406,120.5,organic_search
2,2013-01-03,10076,7458,36963,0.00401,263.6,direct
3,2013-01-04,9973,8063,53078,0.00562,151.8,direct
4,2013-01-05,10223,7882,36790,0.00525,168.6,referral
5,2013-01-06,9545,6992,47160,0.00438,263.6,social_media
6,2013-01-07,10203,7354,32749,0.00443,252.1,organic_search
7,2013-01-08,9456,7459,31482,0.00440,192.4,paid_search
8,2013-01-09,9162,7108,46717,0.00409,312.8,organic_search
9,2013-01-10,9887,7878,38472,0.00544,285.4,organic_search


In [6]:
products.shape

(2412, 8)

In [7]:
def question1(data):
    #Chuyển dạng của cột order date từ dạng str sang datetime của pandas
    data['order_date'] = pd.to_datetime(data['order_date'])
    #Sắp xếp data theo customer_id, order_date và order_id
    data = data.sort_values(by = ['customer_id', 'order_date','order_id'])
    #Bỏ đi những cột không cần thiết
    new_data = data.drop(['zip','order_status', 'payment_method', 'device_type', 'order_source'],axis = 1)
    #Tạo 1 cột mới để lưu khoảng cách giữa 2 lần đặt hàng liên tiếp
    new_data['inter_order_gap'] = new_data.groupby('customer_id')['order_date'].diff()
    #Bỏ đi NaN và chuyển định dạng về ngày
    gap_days = new_data['inter_order_gap'].dt.days.dropna()
    #Tính số trung vị
    median_gap = gap_days.median()
    print(median_gap)
question1(orders)
    

144.0


In [8]:
def question2(data):
    #Tạo 1 cột mới trong data cho tỷ suất lợi nhuận gộp theo công thức đề bài
    data['margin'] = (data['price'] - data['cogs'])/data['cogs']
    #Tạo 1 dataframe mới với 1 cột là từng loại segment, 1 cột là trung bình cộng của margin mình vừa tính cho từng sản phẩm
    segment_margin = data.groupby('segment', as_index = False)['margin'].mean()
    #Sort để biết segment nào có tỷ suất lợi nhuận gộp trung bình cao nhất
    result = segment_margin.sort_values(by = 'margin', ascending = False)
    print(result)
question2(products)

       segment    margin
6     Standard  0.505893
5      Premium  0.457631
1  All-weather  0.452157
4  Performance  0.421009
0   Activewear  0.420140
2     Balanced  0.403299
7       Trendy  0.378510
3     Everyday  0.363563


In [9]:
def question3(products, returns):
    merged = pd.merge(returns, products, on = 'product_id', how = 'inner')
    streetwear_returns = merged[merged['category'] == 'Streetwear']
    top_reason = streetwear_returns['return_reason'].value_counts()
    print(top_reason)
question3(products,returns)
    

return_reason
wrong_size          7626
defective           4330
not_as_described    3854
changed_mind        3830
late_delivery       2159
Name: count, dtype: int64
